# Earth2Studio grids

Start here before the coordinate-array tutorial. A grid object describes spatial geometry without containing weather data.

## Define a grid object

A grid object is a concrete implementation of the `GridDefinition` interface. The interface makes dimension order, shape, topology, coordinate generation, metadata, and identity explicit.

```python
class GridDefinition(ABC):
    @property
    @abstractmethod
    def dims(self) -> tuple[str, ...]: ...

    @property
    @abstractmethod
    def shape(self) -> tuple[int, ...]: ...

    @property
    @abstractmethod
    def topology(self) -> GridTopology: ...

    @abstractmethod
    def index_coordinates(self) -> xr.Coordinates: ...

    @abstractmethod
    def geographic_coordinates(self, indexes) -> xr.Coordinates: ...

    @abstractmethod
    def to_metadata(self) -> dict[str, object]: ...

    @abstractmethod
    def fingerprint(self) -> str: ...
```

The base interface also provides optional `crs`, `subset_indexers()`, and `cell_bounds()` hooks. Concrete grids implement the geometry-specific behavior.

In [ ]:
import numpy as np
import xarray as xr

import earth2studio as e2s

## Use a known grid

The registry maps stable names and aliases to complete definitions. `resolve_grid()` returns the definition itself, so its interface is discoverable.

In [ ]:
hrrr = e2s.resolve_grid("hrrr")
{
    "type": type(hrrr).__name__,
    "dims": hrrr.dims,
    "shape": hrrr.shape,
    "topology": hrrr.topology,
    "crs": hrrr.crs.name,
}

## Register a projected grid

Choose one of the concrete definitions. A projected grid requires ordered `y` and `x` coordinates plus any CRS accepted by PyProj. Registration adds identity; the definition supplies behavior.

In [ ]:
regional = e2s.ProjectedGrid(
    y=np.arange(3) * 3_000.0,
    x=np.arange(4) * 3_000.0,
    coordinate_reference_system=(
        "+proj=lcc +lat_1=30 +lat_2=60 +lat_0=38 +lon_0=-97 "
        "+datum=WGS84 +units=m +type=crs"
    ),
)
e2s.register_grid("tutorial-lcc", regional)

A registered grid can size a coordinate-only model contract without allocating field values. Geographic coordinates are generated only when requested.

In [ ]:
signature = e2s.coord_array(
    dims=("batch", "variable", "y", "x"),
    coords={"variable": ["u10m"]},
    dynamic=("batch",),
    grid="tutorial-lcc",
)
located = signature.e2s.materialize_grid_coords()
signature.shape, signature.data.nbytes, located.lat.shape

## Infer ordinary Xarray grids

Registration is optional. Separate one-dimensional latitude and longitude coordinates define a rectilinear grid.

In [ ]:
latlon = xr.DataArray(
    np.zeros((2, 3)),
    dims=("lat", "lon"),
    coords={"lat": [40.0, 39.0], "lon": [250.0, 251.0, 252.0]},
)
type(e2s.infer_grid(latlon)).__name__

Two-dimensional `lat` and `lon` coordinates on `y, x` define a curvilinear grid. For arbitrary locations, `x` is the ordered index and `lat` and `lon` are auxiliary coordinates on it.

In [ ]:
points = xr.Dataset(
    coords={
        "x": np.arange(3),
        "lat": ("x", [35.2, 40.8, 51.0]),
        "lon": ("x", [-97.4, -74.0, 0.1]),
    }
)
point_grid = e2s.infer_grid(points)
point_grid.dims, point_grid.topology, point_grid.index_coordinates()

## Select without regridding

Dimension keywords use normal positional indexing. Geographic bounds are translated by the grid into spatial indexers; field data remains outside the grid implementation.

In [ ]:
subset = latlon.e2s.subset(bounds=(-110, 38, -90, 41))
subset.shape, subset.coords

## Regridding readiness

Known and inferred definitions expose the same geographic-coordinate contract. A future regridder can therefore normalize either source into center coordinates, use optional cell geometry for conservative methods, and cache weights with `fingerprint()` without requiring every user grid to be registered.